# GlowWise AI - Google Colab Text CNN Training Workflow 🧠⚡

This notebook provides the **optional Google Colab workflow** to train a real **1D Convolutional Neural Network (CNN)** for predicting skincare review satisfaction (`high_satisfaction`).

### Why Google Colab?
Deep learning architectures like Text CNNs (utilizing word embeddings and 1D convolutions) require significant parallel compute. Training these models on CPU is slow, and local environments (especially Python 3.13) may lack stable TensorFlow binaries.

By running this notebook in **Google Colab** with a **GPU T4 hardware accelerator**, you can execute the deep learning workflow in under a minute.

## 1. Environment & Data Setup

Colab comes pre-installed with TensorFlow. Run the cell below to verify GPU availability and specify data loading.

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_curve, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print('GPU device not found. Proceeding on CPU (highly recommend enabling GPU in Runtime settings).')
else:
    print(f'Found GPU at: {device_name}')

### Load the Skincare Reviews Dataset

Upload the preprocessed dataset `glowwise_reviews_sample_100k.csv` to Colab (drag and drop it into the file explorer on the left) or load it from Google Drive.

In [ ]:
# Search paths for Colab and local environments
data_paths = [
    "glowwise_reviews_sample_100k.csv",
    "/content/glowwise_reviews_sample_100k.csv",
    "/content/drive/MyDrive/glowwise_reviews_sample_100k.csv",
    "../../data/processed/glowwise_reviews_sample_100k.csv",
    "data/processed/glowwise_reviews_sample_100k.csv"
]

data_path = None
for path in data_paths:
    if os.path.exists(path):
        data_path = path
        print(f"Found dataset at: {path}")
        break

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Please upload 'glowwise_reviews_sample_100k.csv' to Google Colab, "
        "or run the preprocessing script locally to generate it."
    )

df = pd.read_csv(data_path)

# Impute and clean text fields
for col in ["review_title", "review_text", "combined_text"]:
    df[col] = df[col].fillna("")

df = df.dropna(subset=["high_satisfaction"])
df = df[df["combined_text"].str.strip() != ""]
print(f"Total cleaned reviews loaded: {len(df):,}")

## 2. Train / Test Split

To make comparisons fair, we apply the exact same split parameters: 80/20 stratified train/test split. 

We use a **15,000 training subset** for the Text CNN (replicating the local experiments) and evaluate on the full **20,000 test set**.

In [ ]:
X = df
y = df["high_satisfaction"].astype(int)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_test_text = X_test_df["combined_text"].astype(str)
print(f"Full Test Set Size: {len(X_test_df):,}")

# Train subset (15k samples)
train_subset_size = 15000
_, X_train_sub, _, y_train_sub = train_test_split(
    X_train_df, y_train, test_size=train_subset_size, stratify=y_train, random_state=42
)
X_train_text = X_train_sub["combined_text"].astype(str)
print(f"Train Subset Size: {len(X_train_text):,}")

# Compute class weights to address label imbalance
classes = np.unique(y_train_sub)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_sub)
class_weight_dict = {c: w for c, w in zip(classes, weights)}
print(f"Computed class weights: {class_weight_dict}")

## 3. Text CNN Architecture & Training

We use Keras `TextVectorization` to map strings to integer sequences. We fit/adapt it **only on the training text subset** to avoid data leakage.

In [ ]:
max_vocab_size = 15000
max_seq_len = 150

vectorizer = TextVectorization(
    max_tokens=max_vocab_size,
    output_sequence_length=max_seq_len,
    output_mode='int'
)

vectorizer.adapt(X_train_text.values)

X_train_seq = vectorizer(X_train_text.values)
X_test_seq = vectorizer(X_test_text.values)
print(f"Vectorized sequences shapes: Train {X_train_seq.shape}, Test {X_test_seq.shape}")

### Build and Train Conv1D Model

The model structure extracts local n-grams using 1D convolutional layers, pooling the most prominent sentiment signals.

In [ ]:
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(max_seq_len,)),
    tf.keras.layers.Embedding(input_dim=max_vocab_size, output_dim=128, input_length=max_seq_len),
    tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

print("Starting Text CNN Training (with GPU acceleration if configured)... ")
t0 = time.time()
history = cnn_model.fit(
    X_train_seq, y_train_sub,
    epochs=8,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)
elapsed = time.time() - t0
print(f"Training finished in {elapsed:.2f} seconds!")

## 4. Evaluation & Diagnostics Curves

Evaluate on the 20,000 test set and generate curves.

In [ ]:
y_prob = cnn_model.predict(X_test_seq).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
p_classes, r_classes, f_classes, support = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall_vals, precision_vals)

cnn_metrics = {
    "accuracy": float(accuracy),
    "macro_f1": float(f_macro),
    "weighted_f1": float(f_weighted),
    "class_0_recall": float(r_classes[0]),
    "class_0_precision": float(p_classes[0]),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "training_time_seconds": float(elapsed)
}

print("=== CNN Metrics Results ===")
print(json.dumps(cnn_metrics, indent=4))

### Plot Training Diagnostics & Confusion Matrix

In [ ]:
# Side-by-side loss and accuracy curves
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5.5))

ax_loss.plot(history.history['loss'], label='Training Loss', color='#3B243B', linewidth=2)
ax_loss.plot(history.history['val_loss'], label='Validation Loss', color='#3B243B', linestyle='--', linewidth=2)
ax_loss.set_xlabel('Epochs', fontweight='bold')
ax_loss.set_ylabel('Loss', fontweight='bold')
ax_loss.set_title('CNN Loss Curve', fontweight='bold')
ax_loss.legend()
ax_loss.grid(True, linestyle='--', alpha=0.5)

ax_acc.plot(history.history['accuracy'], label='Training Accuracy', color='#C39B6F', linewidth=2)
ax_acc.plot(history.history['val_accuracy'], label='Validation Accuracy', color='#C39B6F', linestyle='--', linewidth=2)
ax_acc.set_xlabel('Epochs', fontweight='bold')
ax_acc.set_ylabel('Accuracy', fontweight='bold')
ax_acc.set_title('CNN Accuracy Curve', fontweight='bold')
ax_acc.legend()
ax_acc.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Text CNN Keras Training Curves', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150)
plt.show()

# Confusion Matrix Plotting helper
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Purples)
plt.title('Text CNN Confusion Matrix', fontweight='bold', pad=15)
plt.colorbar()
classes = ['Low/Medium', 'High']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)
plt.xlabel('Predicted Label', fontweight='bold', labelpad=10)
plt.ylabel('True Label', fontweight='bold', labelpad=10)

# Add values
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, format(cm[i, j], ','),
             ha="center", va="center",
             color="white" if cm[i, j] > thresh else "black",
             fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150)
plt.show()

### Save Metrics JSON File

In [ ]:
with open('cnn_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(cnn_metrics, f, indent=4)
print("Metrics saved to cnn_metrics.json!")